# BERT training experiments and final selection
## 1. Introduction
BERT-base-uncased is the first encoder investigated. Controlled runs explore learning
rate and maximum training duration. Selection uses SUM validation data only; neither
SUM test nor UMWP is loaded here. Precision, recall and F1 use UNANSWERABLE as the
positive class. The selected model will later be evaluated on untouched SUM test and
external UMWP data. Epoch budgets below are maxima, subject to early stopping.


## 2. Load experiment results
Read the existing configurations, histories and validation reports. Run 01 has no
standalone config file, so its report's configuration is used. Loading all runs here
does not imply comparing them before their chronological introduction.

Runs 05 and 06 are optional while pending: only complete result artifacts are loaded.
The planned comparison order is Run 01, Run 03, Run 02, Run 04, Run 05, Run 06.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'results/bert/run_01').is_dir())
PLANNED_RUNS = {
    'run_01': (2e-5, 5),
    'run_03': (2e-5, 7),
    'run_02': (3e-5, 7),
    'run_04': (4e-5, 7),
    'run_05': (5e-5, 5),
    'run_06': (5e-5, 7),
}
RUN_NAMES = list(PLANNED_RUNS)
runs = {}
for name in RUN_NAMES:
    folder = ROOT / 'results/bert' / name
    report_path = folder / 'validation_metrics.json'
    history_path = folder / 'training_history.csv'
    if name in {'run_05', 'run_06'} and not all(
        path.is_file() for path in [report_path, history_path, folder / 'config.json']
    ):
        print(name, ': pending; complete training/validation artifacts not available')
        continue
    report = json.loads(report_path.read_text(encoding='utf-8'))
    config_path = folder / 'config.json'
    config = dict(report['configuration'])
    if config_path.exists():
        config.update(json.loads(config_path.read_text(encoding='utf-8')))
    runs[name] = dict(config=config, report=report, history=pd.read_csv(history_path))
    print(name, ': loaded history and validation report;',
          'config.json loaded' if config_path.exists() else 'configuration from validation report')

RUN_NAMES = [name for name in PLANNED_RUNS if name in runs]
pending_runs = [name for name in PLANNED_RUNS if name not in runs]
display(pd.DataFrame([
    dict(run=name, learning_rate=lr, max_epochs=epochs,
         status='available' if name in runs else 'pending')
    for name, (lr, epochs) in PLANNED_RUNS.items()
]).set_index('run'))

def epoch_table(name):
    history = runs[name]['history']
    columns = ['loss', 'eval_loss', 'eval_accuracy', 'eval_precision', 'eval_recall', 'eval_f1']
    # Trainer stores training/evaluation events in separate rows. Exclude final
    # aggregate logs and combine non-null epoch events, without forward filling.
    events = history.loc[history[columns].notna().any(axis=1), ['epoch', *columns]]
    return events.groupby('epoch', sort=True)[columns].last().rename(columns={'loss': 'train_loss'})

def comparison(names):
    rows = []
    for name in names:
        c = runs[name]['config']
        m = runs[name]['report']['best_validation_metrics']
        rows.append(dict(run=name, learning_rate=c['learning_rate'],
                         max_epochs=c['max_epochs'], batch_size=c['batch_size'],
                         **{f'best_validation_{key}': m[f'eval_{key}']
                            for key in ['accuracy', 'precision', 'recall', 'f1']}))
    return pd.DataFrame(rows).set_index('run')

def show_run(name):
    display(pd.Series(runs[name]['config'], name='configuration').to_frame())
    display(epoch_table(name))
    print('Metrics after reloading the validation-selected checkpoint:')
    display(pd.Series(runs[name]['report']['best_validation_metrics']).to_frame('value'))


## 3. Run 01 ? Initial baseline
Run 01 establishes the initial BERT baseline: LR 2e-5, maximum 5 epochs, batch size 32.


In [ ]:
show_run('run_01')


## 4. Run 02 ? First improvement attempt
First inspect Run 02 alone: LR 3e-5, maximum 7 epochs, batch size 32.


In [ ]:
show_run('run_02')


In [ ]:
display(comparison(['run_01', 'run_02']))


Run 02 improved validation F1, but two hyperparameters changed simultaneously:
learning rate 2e-5 ? 3e-5 and maximum epochs 5 ? 7. The improvement cannot yet be
attributed to either change alone. This motivates Run 03.


## 5. Run 03 ? Isolating the epoch effect
Run 03 returns to LR 2e-5 with a 7-epoch budget and batch size 32.


In [ ]:
show_run('run_03')


In [ ]:
display(comparison(['run_01', 'run_03']))


Run 01 versus Run 03 holds LR at 2e-5 and changes the epoch budget from 5 to 7,
isolating extended training more cleanly. The warmup-step count and linear learning-rate
schedule also depend on total planned steps, so this is a duration-budget comparison,
not simply two more epochs appended to an otherwise identical trajectory.


In [ ]:
display(comparison(['run_01', 'run_02', 'run_03']))
f1 = comparison(RUN_NAMES)['best_validation_f1']
display(Markdown(f'At 2e-5, extending the budget changes F1 by **{f1.run_03-f1.run_01:+.6f}**. Run 02 versus Run 03 changes F1 by **{f1.run_02-f1.run_03:+.6f}**.'))


## 6. Run 04 ? Learning-rate direction
Inspect LR 4e-5 with the same 7-epoch budget and batch size 32.


In [ ]:
show_run('run_04')


The three seven-epoch experiments isolate learning-rate direction more directly.


In [ ]:
display(comparison(['run_03', 'run_02', 'run_04']))
display(comparison(RUN_NAMES))


## 6a. Runs 05 and 06 - Exploratory learning-rate completion
Run 05: LR 5e-5, maximum 5 epochs. Run 06: LR 5e-5, maximum 7 epochs.
Both preserve the Run 04 training setup, including early stopping. Results will be
shown only after complete training/validation artifacts are available.


In [ ]:
for name in ['run_05', 'run_06']:
    if name in runs:
        show_run(name)
    else:
        display(Markdown(f'**{name}: pending execution/results.**'))
if 'run_05' in runs and 'run_06' in runs:
    display(comparison(['run_05', 'run_06']))


## 7. Training dynamics
Compare epoch losses and validation F1, using only logged training/validation events.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for name in RUN_NAMES:
    table = epoch_table(name)
    for ax, column, title in zip(axes, ['train_loss', 'eval_loss', 'eval_f1'],
                                  ['Train loss', 'Validation loss', 'Validation F1']):
        values = table[column].dropna()
        ax.plot(values.index, values.values, marker='o', label=name)
        ax.set(xlabel='Epoch', ylabel=title, title=title)
        ax.grid(alpha=0.25)
        ax.legend()
fig.tight_layout()
plt.show()
for name in RUN_NAMES:
    t = epoch_table(name)
    best_epoch = t['eval_f1'].idxmax()
    train = t['train_loss'].dropna()
    val = t['eval_loss'].dropna()
    display(Markdown(f"**{name}**: train loss {train.iloc[0]:.4f} ? {train.iloc[-1]:.4f}; "
                     f"validation loss {val.iloc[0]:.4f} ? {val.iloc[-1]:.4f}; "
                     f"highest epoch validation F1 at epoch {best_epoch:g}."))
    if train.iloc[-1] < train.iloc[0] and val.iloc[-1] > val.min():
        display(Markdown('Training loss falls while validation loss finishes above its minimum: '
                         'this is consistent with possible overfitting. F1 and loss can move '
                         'differently because F1 uses discrete predictions, whereas loss also '
                         'reflects confidence. This observation does not change the selection criterion.'))


## 8. Controlled comparison summary
Highlight the highest validation F1 programmatically; no test metrics are used.


In [ ]:
summary = comparison(RUN_NAMES)
display(summary.style.format(precision=6).highlight_max(subset=['best_validation_f1'], color='#d9ead3'))


## 9. BERT selection from available validation results
Select the highest validation F1 without forcing an expected winner. Selection remains provisional while planned runs are pending. These are single-seed experiments, so small differences do not establish statistical significance.


In [ ]:
selected = summary['best_validation_f1'].idxmax()
c = runs[selected]['config']
model_path = Path('models/bert') / selected / 'best_model'
display(Markdown(f"Selected **{selected}**, validation F1 **{summary.loc[selected, 'best_validation_f1']:.6f}**. "
                 f"Configuration: **{c['model_name']}**, LR **{c['learning_rate']:g}**, "
                 f"maximum **{c['max_epochs']} epochs**, batch size **{c['batch_size']}**. "
                 f"Selected checkpoint: `{model_path.as_posix()}`."))
if pending_runs:
    display(Markdown('**Provisional selection:** awaiting ' + ', '.join(pending_runs) +
                     '. Recompute selection after their validation artifacts are available.'))


## 10. Methodological conclusion
Runs 05 and 06 are exploratory experiments added to complete the learning-rate range
reported in the original BERT study, extending the experiments to 5e-5 with maximum
budgets of 5 and 7 epochs. They have not been executed; no results are assumed.
Hyperparameter/model selection continues to use validation performance only. SUM test
and UMWP results must not be used to select the best BERT run or checkpoint. Untouched
SUM test data is reserved for final in-distribution evaluation; UMWP is an external
generalization benchmark. Any future fresh/private dataset is also evaluation-only.
